# IP 联动评分 MVP v7 Pro Max

这版是 v7 的完整版：保留原来比较细的特征，同时加入时间窗回测。

核心原则：

- **预测分只看联动前数据**
- **活动表现分只看活动期数据**
- **活动后留存分只看活动结束后的残留数据**
- **collab / 联动 / NTEPorsche 这类事件词不进入用户重合度**
- **用户重合只看 core_keywords**
- **活动热度单独算 campaign score**
- **执行、授权、法务仍然是人工业务先验**

这版适合两种场景：

1. 回测已经发生的联动  
2. 用同一套 baseline 标准预测未来候选 IP


## 1. 依赖

In [33]:
import pandas as pd
import numpy as np
import re
import math
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)


## 2. 上传 / 读取 Apify TikTok CSV

如果在 Colab 里跑，可以先取消注释上传文件。

这版默认吃 Apify TikTok Scraper 导出的 All fields CSV。


In [34]:
from google.colab import files
uploaded = files.upload()

csv_path = list(uploaded.keys())[0]

raw_df = pd.read_csv(csv_path)
print("当前读取文件:", csv_path)
print("原始数据量:", raw_df.shape)
display(raw_df.head())

Saving dataset_tiktok-scraper_2026-06-08_03-32-18-107 (1).csv to dataset_tiktok-scraper_2026-06-08_03-32-18-107 (1) (2).csv
当前读取文件: dataset_tiktok-scraper_2026-06-08_03-32-18-107 (1) (2).csv
原始数据量: (20, 96)


,authorMeta/avatar,authorMeta/digg,authorMeta/fans,authorMeta/following,authorMeta/friends,authorMeta/heart,authorMeta/id,authorMeta/name,authorMeta/nickName,authorMeta/originalAvatarUrl,authorMeta/privateAccount,authorMeta/profileUrl,authorMeta/signature,authorMeta/verified,authorMeta/video,collectCount,commentCount,commentsDatasetUrl,createTime,createTimeISO,diggCount,effectStickers/0/ID,effectStickers/0/name,effectStickers/0/stickerStats/useCount,hashtags/0/cover,hashtags/0/id,hashtags/0/name,hashtags/0/title,hashtags/1/cover,hashtags/1/id,hashtags/1/name,hashtags/1/title,hashtags/2/cover,hashtags/2/id,hashtags/2/name,hashtags/2/title,hashtags/3/cover,hashtags/3/id,hashtags/3/name,hashtags/3/title,hashtags/4/cover,hashtags/4/id,hashtags/4/name,hashtags/4/title,hashtags/5/cover,hashtags/5/id,hashtags/5/name,hashtags/5/title,hashtags/6/cover,hashtags/6/id,hashtags/6/name,hashtags/6/title,hashtags/7/cover,hashtags/7/id,hashtags/7/name,hashtags/7/title,id,isAd,isPinned,isSlideshow,musicMeta/coverMediumUrl,musicMeta/musicAlbum,musicMeta/musicAuthor,musicMeta/musicId,musicMeta/musicName,musicMeta/musicOriginal,musicMeta/originalCoverMediumUrl,musicMeta/playUrl,playCount,repostCount,searchQuery,shareCount,text,textLanguage,videoMeta/coverUrl,videoMeta/definition,videoMeta/duration,videoMeta/format,videoMeta/height,videoMeta/originalCoverUrl,videoMeta/subtitleLinks,videoMeta/subtitleLinks/0/downloadLink,videoMeta/subtitleLinks/0/language,videoMeta/subtitleLinks/0/source,videoMeta/subtitleLinks/0/sourceUnabbreviated,videoMeta/subtitleLinks/0/tiktokLink,videoMeta/subtitleLinks/0/version,videoMeta/subtitleLinks/1/downloadLink,videoMeta/subtitleLinks/1/language,videoMeta/subtitleLinks/1/source,videoMeta/subtitleLinks/1/sourceUnabbreviated,videoMeta/subtitleLinks/1/tiktokLink,videoMeta/subtitleLinks/1/version,videoMeta/transcriptionLink,videoMeta/width,webVideoUrl
0,https://p19-common-sign.tiktokcdn-us.com/tos-u...,38,183800,11,0,2800000,7193290802231575598,ntegame.official,NTE Global,https://p19-common-sign.tiktokcdn-us.com/tos-u...,False,https://www.tiktok.com/@ntegame.official,Supernatural urban open-world RPG developed by...,False,102,6183,1010,NaN,1779537600,2026-05-23T12:00:00.000Z,58300,NaN,NaN,NaN,NaN,9073353,nte,NaN,NaN,7391040385734148127,nevernesstoeverness,NaN,NaN,7632240658753552404,nteporsche,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7643007206341889311,False,False,False,https://p19-common-sign.tiktokcdn-us.com/tos-u...,NaN,NTE Global,7643007181708708638,原聲 - NTE Global,True,https://p19-common-sign.tiktokcdn-us.com/tos-u...,https://v16-webapp-prime.us.tiktok.com/video/t...,409400,0,NTE,6726,NTE丨Porsche Collab Coming Soon!\n\nNTE丨Porsche...,en,https://p19-common-sign.tiktokcdn-us.com/tos-u...,720p,54,mp4,720,https://p19-common-sign.tiktokcdn-us.com/tos-u...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1280,https://www.tiktok.com/@ntegame.official/video...
1,https://p16-common-sign.tiktokcdn-us.com/tos-u...,11500,348600,106,0,32600000,7017086837139932165,yizarrhh,YizarrH🌸𓁹‿𓁹,https://p16-common-sign.tiktokcdn-us.com/tos-u...,False,https://www.tiktok.com/@yizarrhh,Gaming Content Creator,False,6533,8806,520,NaN,1778841530,2026-05-15T10:38:50.000Z,59200,NaN,NaN,NaN,NaN,7138408407446323227,yizarr🌸,NaN,NaN,1637342470396934,fypシ,NaN,NaN,88764338,foryoupage,NaN,NaN,7.391040e+18,nevernesstoeverness,NaN,NaN,9073353.0,nte,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7640065913928043790,False,False,False,https://p16-common-sign.tiktokcdn-us.com/tos-u...,NaN,YizarrH🌸𓁹‿𓁹,7640066133860682509,original sound - YizarrH🌸𓁹‿𓁹,True,https://p16-common-sign.tiktokcdn-us.com/tos-u...,https://v16-webapp-prime.us.tiktok.com/video/t...,370200,0,NTE,13900,#yizarr🌸 #fypシ #foryoupage #nevernesstoevernes...,un,https://p19-common-sign.tiktokcdn-us.com/tos-u...,540p,15,mp4,1024,https://p19-common-sign.tiktokcdn-us.com/tos-u...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,576,https://www.t

## 3. 配置区

这里是最常改的地方。

### 关键词拆分

- `core_keywords`：长期基础兴趣，用于用户重合
- `event_keywords`：活动/联动事件热度，不进用户重合
- `purchase_keywords`：付费、抽卡、购买意图
- `risk_keywords`：违和、硬塞、负面风险
- `ugc_keywords`：二创、剪辑、meme、传播潜力

### 重点修正

不要把 `collab`、`联动`、`NTEPorsche` 放进 `core_keywords`。


In [35]:
base_config = {
    "interest": "NTE",
    "core_keywords": [
        "NTE",
        "Neverness to Everness",
        "NevernesstoEverness",
        "异环",
        "ntegameplay",
        "NTEcreator"
    ],
    "event_keywords": [
        "collab", "crossover", "联动", "合作",
        "NTE Porsche", "NTEPorsche", "Porsche Collab", "coming soon"
    ],
    "channels": [
        "nte",
        "neverness-to-everness",
        "异环",
        "tiktok_search_nte"
    ]
}

target_ips = [
    {
        "interest": "Porsche",

        # 长期兴趣词：用于预测和 baseline 用户重合
        "core_keywords": [
            "Porsche", "保时捷", "911", "Taycan", "Panamera", "Carrera"
        ],

        # 活动事件词：用于 campaign heat，不进入用户重合
        "event_keywords": [
            "NTE Porsche", "NTEPorsche", "Porsche Collab",
            "保时捷联动", "collab", "crossover", "联动", "coming soon"
        ],

        "channels": [
            "porsche",
            "tiktok_search_porsche"
        ]
    }
]

purchase_keywords = [
    "pull", "pulled", "gacha", "skin", "banner", "limited", "buy", "bundle", "top up",
    "抽", "卡池", "皮肤", "限定", "礼包", "氪", "买", "入手"
]

positive_keywords = [
    "good", "great", "love", "like", "hype", "perfect", "cool", "amazing", "iconic", "finally",
    "喜欢", "期待", "神", "好看", "帅", "酷", "香"
]

negative_keywords = [
    "bad", "hate", "trash", "boring", "uninstalled", "scam", "boycott", "disappointed",
    "烂", "讨厌", "卸载", "抵制", "失望", "垃圾"
]

risk_keywords = [
    "forced", "weird", "out of place", "doesn't fit", "cash grab", "cringe",
    "违和", "硬塞", "不搭", "出戏", "割韭菜"
]

ugc_keywords = [
    "fanart", "cosplay", "meme", "edit", "clip", "amv", "showcase", "reaction", "guide",
    "二创", "cos", "梗图", "剪辑", "攻略", "展示"
]

creator_keywords = [
    "creator", "streamer", "youtuber", "review", "reaction", "guide", "showcase",
    "主播", "up主", "攻略", "测评", "直播"
]

pd.DataFrame(target_ips)


,interest,core_keywords,event_keywords,channels
0,Porsche,"[Porsche, 保时捷, 911, Taycan, Panamera, Carrera]","[NTE Porsche, NTEPorsche, Porsche Collab, 保时捷联...","[porsche, tiktok_search_porsche]"


## 4. 时间窗设置

回测的关键是时间窗。

比如 Porsche 联动在 `2026-06-03` 上线，那：

- `baseline`：2026-06-03 之前，只用于预测
- `campaign`：上线后 14 天，只用于活动复盘
- `post`：活动期结束后，看衰减和长期残留

如果以后评估别的 IP，只需要改 `campaign_date`。


In [36]:
campaign_date = pd.Timestamp("2026-06-03", tz="UTC")

baseline_start = None          # 例如 pd.Timestamp("2026-03-01", tz="UTC")；None 表示不设开始时间
campaign_days = 14
post_days = 60

print("活动开始时间:", campaign_date)
print("活动期天数:", campaign_days)
print("活动后观察天数:", post_days)


活动开始时间: 2026-06-03 00:00:00+00:00
活动期天数: 14
活动后观察天数: 60


## 5. 通用函数

In [37]:
def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).lower()

def safe_num(x, default=0):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default

def hit_count(text, keywords):
    text = norm_text(text)
    total = 0
    for kw in keywords:
        kw = str(kw).strip().lower()
        if kw:
            total += len(re.findall(re.escape(kw), text))
    return total

def has_any(text, keywords):
    return hit_count(text, keywords) > 0

def pct(x):
    try:
        return f"{float(x) * 100:.1f}%"
    except Exception:
        return "-"

def score_0_100(s):
    s = pd.Series(s).astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)
    if len(s) == 0:
        return s
    if s.max() == s.min():
        return pd.Series([50 if s.max() > 0 else 0] * len(s), index=s.index)
    return (s - s.min()) / (s.max() - s.min()) * 100

def log_score_0_100(s):
    return score_0_100(np.log1p(pd.Series(s).astype(float).fillna(0).clip(lower=0)))

def grade(score):
    if score >= 85:
        return "S"
    if score >= 75:
        return "A"
    if score >= 65:
        return "B"
    if score >= 50:
        return "C"
    return "D"

def action_for_grade(g):
    return {
        "S": "强烈建议优先推进",
        "A": "建议重点推进",
        "B": "可作为备选观察",
        "C": "低优先级，仅低成本时考虑",
        "D": "暂不建议推进"
    }.get(g, "需人工复核")

def risk_penalty(x):
    x = str(x).lower()
    if x in ["high", "高"]:
        return 35
    if x in ["mid", "medium", "中"]:
        return 15
    return 0

def safe_div(a, b):
    return a / b if b and b != 0 else 0


## 6. 标准化 TikTok 数据

把 Apify 字段整理成统一表。

这一步会保留：

- 作者
- 文案
- 播放
- 点赞
- 评论数
- 分享
- 收藏
- 发布时间
- 作者粉丝数
- 视频链接


In [38]:
def build_content_df(raw):
    df = pd.DataFrame({
        "content_id": "tiktok:" + raw["id"].astype(str) if "id" in raw.columns else [f"tiktok:{i}" for i in range(len(raw))],
        "user_id": "tiktok:" + raw["authorMeta/id"].astype(str) if "authorMeta/id" in raw.columns else "tiktok:" + raw.get("authorMeta/name", pd.Series(range(len(raw)))).astype(str),
        "platform": "tiktok",
        "search_query": raw["searchQuery"] if "searchQuery" in raw.columns else "",
        "channel": "tiktok_search_" + raw["searchQuery"].astype(str).str.lower() if "searchQuery" in raw.columns else "tiktok_search_unknown",
        "text": raw["text"].fillna("") if "text" in raw.columns else "",
        "likes": pd.to_numeric(raw["diggCount"], errors="coerce").fillna(0) if "diggCount" in raw.columns else 0,
        "comments": pd.to_numeric(raw["commentCount"], errors="coerce").fillna(0) if "commentCount" in raw.columns else 0,
        "shares": pd.to_numeric(raw["shareCount"], errors="coerce").fillna(0) if "shareCount" in raw.columns else 0,
        "collects": pd.to_numeric(raw["collectCount"], errors="coerce").fillna(0) if "collectCount" in raw.columns else 0,
        "views": pd.to_numeric(raw["playCount"], errors="coerce").fillna(0) if "playCount" in raw.columns else 0,
        "created_at": raw["createTimeISO"] if "createTimeISO" in raw.columns else None,
        "source": raw["webVideoUrl"] if "webVideoUrl" in raw.columns else "apify_tiktok",
        "language": raw["textLanguage"] if "textLanguage" in raw.columns else None,
        "author_name": raw["authorMeta/name"] if "authorMeta/name" in raw.columns else None,
        "author_fans": pd.to_numeric(raw["authorMeta/fans"], errors="coerce").fillna(0) if "authorMeta/fans" in raw.columns else 0,
        "author_video_count": pd.to_numeric(raw["authorMeta/video"], errors="coerce").fillna(0) if "authorMeta/video" in raw.columns else 0,
    })

    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    df["engagement"] = (
        df["likes"]
        + df["comments"] * 2
        + df["shares"] * 3
        + df["collects"] * 2
        + df["views"] * 0.01
    )
    df["engagement_rate"] = (df["likes"] + df["comments"] + df["shares"]) / df["views"].replace(0, np.nan)
    df["engagement_rate"] = df["engagement_rate"].fillna(0)

    return df

content_df = build_content_df(raw_df)

print("标准化后数据量:", content_df.shape)
display(content_df.head())


标准化后数据量: (20, 19)


,content_id,user_id,platform,search_query,channel,text,likes,comments,shares,collects,views,created_at,source,language,author_name,author_fans,author_video_count,engagement,engagement_rate
0,tiktok:7643007206341889311,tiktok:7193290802231575598,tiktok,NTE,tiktok_search_nte,NTE丨Porsche Collab Coming Soon!\n\nNTE丨Porsche...,58300,1010,6726,6183,409400,2026-05-23 12:00:00+00:00,https://www.tiktok.com/@ntegame.official/video...,en,ntegame.official,183800,102,96958.0,0.161299
1,tiktok:7640065913928043790,tiktok:7017086837139932165,tiktok,NTE,tiktok_search_nte,#yizarr🌸 #fypシ #foryoupage #nevernesstoevernes...,59200,520,13900,8806,370200,2026-05-15 10:38:50+00:00,https://www.tiktok.com/@yizarrhh/video/7640065...,un,yizarrhh,348600,6533,123254.0,0.198865
2,tiktok:7644995636294716702,tiktok:7443886183159645230,tiktok,NTE,tiktok_search_nte,вступайте в дс:3\n#winetempress #nevernesstoev...,28900,258,7062,4725,217500,2026-05-28 17:27:49+00:00,https://www.tiktok.com/@winetempress/video/764...,ru,winetempress,275300,11900,62227.0,0.166529
3,tiktok:7647440166813961480,tiktok:6937123059149833217,tiktok,NTE,tiktok_search_nte,She's literally me after exercising a minute.....,38500,287,5426,5623,241800,2026-06-04 07:33:44+00:00,https://www.tiktok.com/@bluebriell/video/76474...,en,bluebriell,40600,19,69016.0,0.182849
4,tiktok:7638288966717869319,tiktok:7246161284314842117,tiktok,NTE,tiktok_search_nte,#Nanally #mint #nte #NTE #gamer,3025,82,373,1156,84500,2026-05-10 15:42:23+00:00,https://www.tiktok.com/@xavi_hashibira23/video...,un,xavi_hashibira23,70800,4465,7465.0,0.041183


## 7. 生成 Creator / KOL 表

这张表用于估计创作者传播力。

现在 TikTok-only 的情况下，它只是一个弱版 Creator 数据。  
后面如果接 Tubular / Nox，可以把它替换或合并进来。


In [39]:
def build_creator_df(content_df, default_related_interest="NTE"):
    creator_df = (
        content_df.groupby(["user_id", "author_name"], dropna=False)
        .agg(
            followers_count=("author_fans", "max"),
            avg_views=("views", "mean"),
            total_views=("views", "sum"),
            avg_engagement_rate=("engagement_rate", "mean"),
            video_count=("author_video_count", "max"),
            scraped_video_count=("content_id", "count"),
            total_engagement=("engagement", "sum")
        )
        .reset_index()
        .rename(columns={"user_id": "creator_id", "author_name": "creator_name"})
    )
    creator_df["platform"] = "TikTok"
    creator_df["related_interest"] = default_related_interest
    return creator_df

creator_df = build_creator_df(content_df, default_related_interest="NTE")
display(creator_df.head())


,creator_id,creator_name,followers_count,avg_views,total_views,avg_engagement_rate,video_count,scraped_video_count,total_engagement,platform,related_interest
0,tiktok:6768284447810257925,senkuzumaki,729,285200.0,285200,0.098299,179,1,45746.0,TikTok,NTE
1,tiktok:6783506479464580101,twilight._stxrmy,103700,60500.0,60500,0.029074,1640,1,2920.0,TikTok,NTE
2,tiktok:6848565161352283141,dark.pandemonium,7857,172700.0,172700,0.241621,155,1,64339.0,TikTok,NTE
3,tiktok:6937123059149833217,bluebriell,40600,241800.0,241800,0.182849,19,1,69016.0,TikTok,NTE
4,tiktok:6943054209126958081,chyue_23,480,136800.0,136800,0.078845,110,1,16923.0,TikTok,NTE


## 8. 切时间窗

这里会生成：

- `baseline_df`
- `campaign_df`
- `post_df`

如果 `baseline_df` 是 0，说明这批数据无法做严格的联动前预测回测。


In [40]:
if baseline_start is None:
    baseline_df = content_df[content_df["created_at"] < campaign_date].copy()
else:
    baseline_df = content_df[
        (content_df["created_at"] >= baseline_start) &
        (content_df["created_at"] < campaign_date)
    ].copy()

campaign_end = campaign_date + pd.Timedelta(days=campaign_days)
post_end = campaign_end + pd.Timedelta(days=post_days)

campaign_df = content_df[
    (content_df["created_at"] >= campaign_date) &
    (content_df["created_at"] < campaign_end)
].copy()

post_df = content_df[
    (content_df["created_at"] >= campaign_end) &
    (content_df["created_at"] < post_end)
].copy()

window_summary = pd.DataFrame([
    {"窗口": "baseline 联动前", "开始": baseline_start if baseline_start is not None else "不限", "结束": campaign_date, "数据量": len(baseline_df)},
    {"窗口": "campaign 活动期", "开始": campaign_date, "结束": campaign_end, "数据量": len(campaign_df)},
    {"窗口": "post 活动后", "开始": campaign_end, "结束": post_end, "数据量": len(post_df)},
])

display(window_summary)


,窗口,开始,结束,数据量
0,baseline 联动前,不限,2026-06-03 00:00:00+00:00,9
1,campaign 活动期,2026-06-03 00:00:00+00:00,2026-06-17 00:00:00+00:00,11
2,post 活动后,2026-06-17 00:00:00+00:00,2026-08-16 00:00:00+00:00,0


## 9. 用户兴趣信号：只用 core_keywords

这一块是 baseline 用户重合的基础。

这里不会用 event keywords。


In [41]:
def infer_core_interest(df, base_config, target_ips):
    rows = []
    configs = [base_config] + target_ips

    for _, r in df.iterrows():
        text = r["text"]
        user_id = r["user_id"]
        strength = 1 + math.log1p(max(0, safe_num(r.get("engagement", 0))))

        for cfg in configs:
            hits = hit_count(text, cfg.get("core_keywords", []))
            if hits > 0:
                rows.append({
                    "user_id": user_id,
                    "interest": cfg["interest"],
                    "signal_type": "core_keyword_mention",
                    "strength": strength * hits,
                    "content_id": r.get("content_id"),
                    "text": text,
                    "created_at": r.get("created_at"),
                    "source": r.get("source")
                })
    return pd.DataFrame(rows)

baseline_signal_df = infer_core_interest(baseline_df, base_config, target_ips)

display(baseline_signal_df.head())
print("baseline signal counts:")
display(baseline_signal_df["interest"].value_counts() if not baseline_signal_df.empty else "No baseline signals")


,user_id,interest,signal_type,strength,content_id,text,created_at,source
0,tiktok:7193290802231575598,NTE,core_keyword_mention,87.374304,tiktok:7643007206341889311,NTE丨Porsche Collab Coming Soon!\n\nNTE丨Porsche...,2026-05-23 12:00:00+00:00,https://www.tiktok.com/@ntegame.official/video...
1,tiktok:7193290802231575598,Porsche,core_keyword_mention,37.446130,tiktok:7643007206341889311,NTE丨Porsche Collab Coming Soon!\n\nNTE丨Porsche...,2026-05-23 12:00:00+00:00,https://www.tiktok.com/@ntegame.official/video...
2,tiktok:7017086837139932165,NTE,core_keyword_mention,25.444021,tiktok:7640065913928043790,#yizarr🌸 #fypシ #foryoupage #nevernesstoevernes...,2026-05-15 10:38:50+00:00,https://www.tiktok.com/@yizarrhh/video/7640065...
3,tiktok:7443886183159645230,NTE,core_keyword_mention,24.077121,tiktok:7644995636294716702,вступайте в дс:3\n#winetempress #nevernesstoev...,2026-05-28 17:27:49+00:00,https://www.tiktok.com/@winetempress/video/764...
4,tiktok:7246161284314842117,NTE,core_keyword_mention,19.836229,tiktok:7638288966717869319,#Nanally #mint #nte #NTE #gamer,2026-05-10 15:42:23+00:00,https://www.tiktok.com/@xavi_hashibira23/video...


baseline signal counts:


,count
interest,
NTE,9
Porsche,1


## 10. 用户强度 / 核心用户 / 高互动用户

这里会把用户分成：

- 普通兴趣用户
- 核心用户
- 高互动用户

这些都只用于联动前 baseline。


In [42]:
def build_user_strength(signal_df):
    if signal_df.empty:
        return pd.DataFrame(columns=[
            "user_id", "interest", "raw_strength", "mention_count",
            "interest_strength_score", "is_active_user", "is_core_user", "is_high_engagement_user"
        ])

    g = (
        signal_df.groupby(["user_id", "interest"], as_index=False)
        .agg(
            raw_strength=("strength", "sum"),
            mention_count=("strength", "count")
        )
    )

    out = []
    for interest, sub in g.groupby("interest"):
        sub = sub.copy()
        sub["interest_strength_score"] = log_score_0_100(sub["raw_strength"] + sub["mention_count"])

        active_cut = max(10, sub["interest_strength_score"].quantile(0.60))
        core_cut = max(20, sub["interest_strength_score"].quantile(0.80))
        high_cut = max(30, sub["interest_strength_score"].quantile(0.90))

        sub["is_active_user"] = sub["interest_strength_score"] >= active_cut
        sub["is_core_user"] = sub["interest_strength_score"] >= core_cut
        sub["is_high_engagement_user"] = sub["interest_strength_score"] >= high_cut

        sub["active_cutoff"] = active_cut
        sub["core_cutoff"] = core_cut
        sub["high_engagement_cutoff"] = high_cut
        out.append(sub)

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

baseline_user_strength_df = build_user_strength(baseline_signal_df)

display(baseline_user_strength_df)


,user_id,interest,raw_strength,mention_count,interest_strength_score,is_active_user,is_core_user,is_high_engagement_user,active_cutoff,core_cutoff,high_engagement_cutoff
0,tiktok:6768284447810257925,NTE,23.461763,1,10.899838,False,False,False,20.243291,55.501105,71.365079
1,tiktok:6848565161352283141,NTE,24.143874,1,12.775787,False,False,False,20.243291,55.501105,71.365079
2,tiktok:7017086837139932165,NTE,47.216602,2,59.092971,True,True,False,20.243291,55.501105,71.365079
3,tiktok:7193290802231575598,NTE,87.374304,1,100.000000,True,True,True,20.243291,55.501105,71.365079
4,tiktok:7246161284314842117,NTE,19.836229,1,0.000000,False,False,False,20.243291,55.501105,71.365079
5,tiktok:7402375761716413445,NTE,42.247490,1,50.113307,True,False,False,20.243291,55.501105,71.365079
6,tiktok:7443886183159645230,NTE,24.077121,1,12.594376,False,False,False,20.243291,55.501105,71.365079
7,tiktok:7593455770780320782,NTE,20.517308,1,2.179423,False,False,False,20.243291,55.501105,71.365079
8,tiktok:7193290802231575598,Porsche,37.446130,1,50.000000,True,True,True,50.000000,50.000000,50.000000


## 11. Baseline 用户重合特征

这一块是预测分里最重要的一块。

包括：

- 基础用户重合
- 活跃用户重合
- 核心用户重合
- 加权核心用户重合
- 高互动用户重合
- Lift


In [43]:
def users_for(df, interest, flag=None):
    sub = df[df["interest"] == interest]
    if flag:
        sub = sub[sub[flag] == True]
    return set(sub["user_id"])

def weighted_core_overlap(user_strength_df, base_interest, target_interest):
    base_core_df = user_strength_df[
        (user_strength_df["interest"] == base_interest) &
        (user_strength_df["is_core_user"])
    ].copy()
    target_core = users_for(user_strength_df, target_interest, "is_core_user")

    if base_core_df.empty or base_core_df["interest_strength_score"].sum() <= 0:
        return 0

    return (
        base_core_df[base_core_df["user_id"].isin(target_core)]["interest_strength_score"].sum()
        / base_core_df["interest_strength_score"].sum()
    )

def overlap_rate(a, b):
    return len(a & b) / len(a) if len(a) else 0

def jaccard(a, b):
    return len(a & b) / len(a | b) if len(a | b) else 0

def baseline_overlap_features(user_strength_df, base_interest, target_interest):
    all_users = set(user_strength_df["user_id"])

    base_all = users_for(user_strength_df, base_interest)
    target_all = users_for(user_strength_df, target_interest)

    base_active = users_for(user_strength_df, base_interest, "is_active_user")
    target_active = users_for(user_strength_df, target_interest, "is_active_user")

    base_core = users_for(user_strength_df, base_interest, "is_core_user")
    target_core = users_for(user_strength_df, target_interest, "is_core_user")

    base_high = users_for(user_strength_df, base_interest, "is_high_engagement_user")
    target_high = users_for(user_strength_df, target_interest, "is_high_engagement_user")

    core_overlap = overlap_rate(base_core, target_core)
    target_core_base_rate = len(target_core) / len(all_users) if all_users else 0
    lift = core_overlap / target_core_base_rate if target_core_base_rate > 0 else 0

    return {
        "target_interest": target_interest,

        "base_user_count": len(base_all),
        "target_user_count": len(target_all),
        "basic_overlap_count": len(base_all & target_all),
        "basic_overlap_rate": overlap_rate(base_all, target_all),
        "basic_jaccard": jaccard(base_all, target_all),

        "active_overlap_count": len(base_active & target_active),
        "active_overlap_rate": overlap_rate(base_active, target_active),

        "core_overlap_count": len(base_core & target_core),
        "core_overlap_rate": core_overlap,
        "weighted_core_overlap_rate": weighted_core_overlap(user_strength_df, base_interest, target_interest),

        "high_engagement_overlap_count": len(base_high & target_high),
        "high_engagement_overlap_rate": overlap_rate(base_high, target_high),

        "target_core_base_rate": target_core_base_rate,
        "lift_core_vs_base": lift,

        "base_core_count": len(base_core),
        "target_core_count": len(target_core),
        "base_high_count": len(base_high),
        "target_high_count": len(target_high)
    }

baseline_overlap_df = pd.DataFrame([
    baseline_overlap_features(baseline_user_strength_df, base_config["interest"], ip["interest"])
    for ip in target_ips
])

display(baseline_overlap_df)


,target_interest,base_user_count,target_user_count,basic_overlap_count,basic_overlap_rate,basic_jaccard,active_overlap_count,active_overlap_rate,core_overlap_count,core_overlap_rate,weighted_core_overlap_rate,high_engagement_overlap_count,high_engagement_overlap_rate,target_core_base_rate,lift_core_vs_base,base_core_count,target_core_count,base_high_count,target_high_count
0,Porsche,8,1,1,0.125,0.125,1,0.333333,1,0.5,0.628563,1,1.0,0.125,4.0,2,1,1,1


## 12. 时间窗文本特征

这里分开算 baseline / campaign / post 三个窗口的文本特征。

注意：

- baseline 可以看 core hit、购买意图、负面、二创，但不能用 event 词去拉高用户重合
- campaign 可以看 event hit，评估活动表现
- post 用来看活动后有没有残留


In [44]:
def text_features_for_window(df, target_ip, window_name):
    target = target_ip["interest"]
    core_kw = target_ip.get("core_keywords", [])
    event_kw = target_ip.get("event_keywords", [])

    rows = []
    for _, r in df.iterrows():
        text = r["text"]
        core_hit = has_any(text, core_kw)
        event_hit = has_any(text, event_kw)
        relevant = core_hit or event_hit

        rows.append({
            "window": window_name,
            "target_interest": target,
            "content_id": r["content_id"],
            "user_id": r["user_id"],
            "core_hit": core_hit,
            "event_hit": event_hit,
            "relevant": relevant,
            "purchase_hit": has_any(text, purchase_keywords),
            "positive_hit": has_any(text, positive_keywords),
            "negative_hit": has_any(text, negative_keywords),
            "risk_hit": has_any(text, risk_keywords),
            "ugc_hit": has_any(text, ugc_keywords),
            "creator_discussion_hit": has_any(text, creator_keywords),
            "likes": safe_num(r["likes"]),
            "comments": safe_num(r["comments"]),
            "shares": safe_num(r["shares"]),
            "collects": safe_num(r["collects"]),
            "views": safe_num(r["views"]),
            "engagement": safe_num(r["engagement"]),
            "author_fans": safe_num(r["author_fans"]),
            "text": text,
            "source": r["source"]
        })

    detail = pd.DataFrame(rows)

    if detail.empty:
        return pd.DataFrame([{
            "target_interest": target,
            "window": window_name,
            "content_count": 0,
            "relevant_content_count": 0,
            "core_hit_rate": 0,
            "event_hit_rate": 0,
            "purchase_intent_rate": 0,
            "positive_rate": 0,
            "negative_rate": 0,
            "fit_risk_rate": 0,
            "ugc_rate": 0,
            "creator_discussion_rate": 0,
            "total_engagement": 0,
            "total_views": 0,
            "total_shares": 0,
            "total_author_fans": 0
        }]), detail

    relevant_df = detail[detail["relevant"]].copy()

    if relevant_df.empty:
        agg = {
            "target_interest": target,
            "window": window_name,
            "content_count": len(detail),
            "relevant_content_count": 0,
            "core_hit_rate": 0,
            "event_hit_rate": 0,
            "purchase_intent_rate": 0,
            "positive_rate": 0,
            "negative_rate": 0,
            "fit_risk_rate": 0,
            "ugc_rate": 0,
            "creator_discussion_rate": 0,
            "total_engagement": 0,
            "total_views": 0,
            "total_shares": 0,
            "total_author_fans": 0
        }
    else:
        agg = {
            "target_interest": target,
            "window": window_name,
            "content_count": len(detail),
            "relevant_content_count": len(relevant_df),
            "core_hit_rate": relevant_df["core_hit"].mean(),
            "event_hit_rate": relevant_df["event_hit"].mean(),
            "purchase_intent_rate": relevant_df["purchase_hit"].mean(),
            "positive_rate": relevant_df["positive_hit"].mean(),
            "negative_rate": relevant_df["negative_hit"].mean(),
            "fit_risk_rate": relevant_df["risk_hit"].mean(),
            "ugc_rate": relevant_df["ugc_hit"].mean(),
            "creator_discussion_rate": relevant_df["creator_discussion_hit"].mean(),
            "total_engagement": relevant_df["engagement"].sum(),
            "total_views": relevant_df["views"].sum(),
            "total_shares": relevant_df["shares"].sum(),
            "total_author_fans": relevant_df["author_fans"].sum()
        }

    return pd.DataFrame([agg]), detail

feature_tables = []
detail_tables = []

for ip in target_ips:
    for df, window_name in [
        (baseline_df, "baseline"),
        (campaign_df, "campaign"),
        (post_df, "post")
    ]:
        agg, detail = text_features_for_window(df, ip, window_name)
        feature_tables.append(agg)
        detail_tables.append(detail)

window_feature_df = pd.concat(feature_tables, ignore_index=True)
window_detail_df = pd.concat(detail_tables, ignore_index=True)

display(window_feature_df)
display(window_detail_df[window_detail_df[["core_hit", "event_hit", "purchase_hit", "negative_hit", "risk_hit", "ugc_hit", "creator_discussion_hit"]].any(axis=1)].head(30))


,target_interest,window,content_count,relevant_content_count,core_hit_rate,event_hit_rate,purchase_intent_rate,positive_rate,negative_rate,fit_risk_rate,ugc_rate,creator_discussion_rate,total_engagement,total_views,total_shares,total_author_fans
0,Porsche,baseline,9,1,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,96958.0,409400.0,6726.0,183800.0
1,Porsche,campaign,11,1,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,11545.0,89800.0,733.0,43500.0
2,Porsche,post,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,window,target_interest,content_id,user_id,core_hit,event_hit,relevant,purchase_hit,positive_hit,negative_hit,risk_hit,ugc_hit,creator_discussion_hit,likes,comments,shares,collects,views,engagement,author_fans,text,source
0,baseline,Porsche,tiktok:7643007206341889311,tiktok:7193290802231575598,True,True,True,True,False,False,False,False,False,58300.0,1010.0,6726.0,6183.0,409400.0,96958.0,183800.0,NTE丨Porsche Collab Coming Soon!\n\nNTE丨Porsche...,https://www.tiktok.com/@ntegame.official/video...
4,baseline,Porsche,tiktok:7645115236218965269,tiktok:6768284447810257925,False,False,False,False,False,False,False,False,True,22500.0,973.0,4562.0,2381.0,285200.0,45746.0,729.0,I unistalled 🥀 #nte #wutheringwaves #neverness...,https://www.tiktok.com/@senkuzumaki/video/7645...
5,baseline,Porsche,tiktok:7638600782807354642,tiktok:7402375761716413445,False,False,False,False,False,False,False,False,True,892.0,17.0,47.0,233.0,18500.0,1718.0,49200.0,Nanally found invisible wall 😱 | Neverness to ...,https://www.tiktok.com/@kamoiruka/video/763860...
7,baseline,Porsche,tiktok:7637894421979139335,tiktok:6848565161352283141,False,False,False,False,False,False,False,True,False,38000.0,202.0,3526.0,6815.0,172700.0,64339.0,7857.0,"#BAICANG || такой он потешный, не могу с него ...",https://www.tiktok.com/@dark.pandemonium/video...
13,campaign,Porsche,tiktok:7648038768926215427,tiktok:7476487512728323104,False,False,False,False,False,False,False,True,False,4358.0,102.0,319.0,462.0,42500.0,6868.0,967.0,First time getting it btw #NTE #nevernesstoeve...,https://www.tiktok.com/@starwonnz/video/764803...
15,campaign,Porsche,tiktok:7648563296873745680,tiktok:7336567727727100934,False,False,False,False,False,False,False,False,True,7658.0,121.0,873.0,1324.0,57600.0,13743.0,26100.0,become a CEO in NTE😎\n#nte #nevernesstoevernes...,https://www.tiktok.com/@shinz.yo/video/7648563...
16,campaign,Porsche,tiktok:7647114259901467911,tiktok:7188366931623035910,True,False,True,True,False,False,False,False,True,6398.0,327.0,733.0,698.0,89800.0,11545.0,43500.0,Just got the Porsche in 6 Pull with 0.19% chan...,https://www.tiktok.com/@icy_frosst/video/76471...
18,campaign,Porsche,tiktok:7648243605773094152,tiktok:6982377154411496474,False,False,False,False,True,False,False,True,True,6014.0,302.0,229.0,556.0,75800.0,9175.0,10200.0,Finally Hunter Level 50 and Rip all my resourc...,https://www.tiktok.com/@agnoziaa/video/7648243...


## 13. Creator / KOL 特征

这部分目前用 TikTok 作者数据做弱版估计。

如果之后接 Tubular / Nox，可以把 `creator_df` 换成更完整的 creator 数据。


In [55]:
def creator_features_for_window(df, target_ip, window_name):
    target = target_ip["interest"]
    core_kw = target_ip.get("core_keywords", [])
    event_kw = target_ip.get("event_keywords", [])

    if df.empty:
        return {
            "target_interest": target,
            "window": window_name,
            "creator_count": 0,
            "creator_total_followers": 0,
            "creator_avg_followers": 0,
            "creator_total_views": 0,
            "creator_avg_engagement_rate": 0,
            "creator_overlap_proxy_count": 0,
            "creator_overlap_proxy_rate": 0
        }

    temp = df.copy()
    temp["target_related"] = temp["text"].apply(lambda x: has_any(x, core_kw) or has_any(x, event_kw))
    temp["base_related"] = temp["text"].apply(lambda x: has_any(x, base_config.get("core_keywords", [])))

    creators = temp[temp["target_related"]].groupby("user_id").agg(
        followers=("author_fans", "max"),
        views=("views", "sum"),
        engagement_rate=("engagement_rate", "mean"),
        videos=("content_id", "count"),
        base_related_any=("base_related", "max")
    ).reset_index()

    if creators.empty:
        return {
            "target_interest": target,
            "window": window_name,
            "creator_count": 0,
            "creator_total_followers": 0,
            "creator_avg_followers": 0,
            "creator_total_views": 0,
            "creator_avg_engagement_rate": 0,
            "creator_overlap_proxy_count": 0,
            "creator_overlap_proxy_rate": 0
        }

    return {
        "target_interest": target,
        "window": window_name,
        "creator_count": len(creators),
        "creator_total_followers": creators["followers"].sum(),
        "creator_avg_followers": creators["followers"].mean(),
        "creator_total_views": creators["views"].sum(),
        "creator_avg_engagement_rate": creators["engagement_rate"].mean(),
        "creator_overlap_proxy_count": int(creators["base_related_any"].sum()),
        "creator_overlap_proxy_rate": creators["base_related_any"].mean()
    }

creator_feature_df = pd.DataFrame([
    creator_features_for_window(df, ip, window_name)
    for ip in target_ips
    for df, window_name in [
        (baseline_df, "baseline"),
        (campaign_df, "campaign"),
        (post_df, "post")
    ]
])

display(creator_feature_df)


,target_interest,window,creator_count,creator_total_followers,creator_avg_followers,creator_total_views,creator_avg_engagement_rate,creator_overlap_proxy_count,creator_overlap_proxy_rate
0,Porsche,baseline,1,183800,183800.0,409400,0.161299,1,1.0
1,Porsche,campaign,1,43500,43500.0,89800,0.083051,1,1.0
2,Porsche,post,0,0,0.0,0,0.000000,0,0.0


In [58]:
# =========================
# 13.5 终端式录入：manual_prior + human_scores
# =========================
# 用法：
# 1. 运行这个 cell
# 2. 它会生成 ip_score_input_cli.py
# 3. 然后自动启动录入
# 4. 录完后会生成 manual_prior.csv 和 human_scores.csv
# 5. 后面的第 14、15 格会自动读取这两个 CSV

cli_code = r'''
import pandas as pd
from pathlib import Path

MANUAL_PRIOR_PATH = Path("manual_prior.csv")
HUMAN_SCORES_PATH = Path("human_scores.csv")

MANUAL_PRIOR_COLUMNS = [
    "target_interest",
    "content_fit_prior",
    "execution_prior",
    "budget_risk",
    "legal_risk",
    "competitor_used"
]

HUMAN_SCORE_COLUMNS = [
    "target_interest",
    "evaluator",
    "role",
    "人工用户匹配",
    "人工内容适配",
    "人工商业潜力",
    "人工传播潜力",
    "人工风险安全",
    "人工执行可行",
    "confidence"
]

ROLE_OPTIONS = {
    "market": "市场",
    "product": "产品",
    "bizdev": "商务",
    "legal": "法务",
    "finance": "财务",
    "community": "社区",
    "default": "其他"
}

HUMAN_SCORE_FIELDS = [
    "人工用户匹配",
    "人工内容适配",
    "人工商业潜力",
    "人工传播潜力",
    "人工风险安全",
    "人工执行可行"
]


def ask_yes_no(prompt, default=False):
    suffix = " [Y/n]: " if default else " [y/N]: "
    value = input(prompt + suffix).strip().lower()

    if value == "":
        return default

    return value in ["y", "yes", "1", "是", "要"]


def ask_text(prompt, default=None):
    value = input(prompt).strip()
    if value == "" and default is not None:
        return default
    return value


def ask_score(prompt, default=None):
    while True:
        value = input(prompt).strip()

        if value == "" and default is not None:
            return default

        try:
            score = float(value)
            if 0 <= score <= 100:
                return score
            print("请输入 0-100 之间的数字。")
        except ValueError:
            print("请输入数字。")


def ask_confidence(prompt="置信度 0-1，默认 0.8：", default=0.8):
    while True:
        value = input(prompt).strip()

        if value == "":
            return default

        try:
            score = float(value)
            if 0 <= score <= 1:
                return score
            print("请输入 0-1 之间的数字。")
        except ValueError:
            print("请输入数字。")


def ask_risk(prompt):
    while True:
        value = input(prompt + " low / mid / high，默认 mid：").strip().lower()
        if value == "":
            return "mid"
        if value in ["low", "mid", "medium", "high"]:
            return "mid" if value == "medium" else value
        print("只能输入 low / mid / high。")


def ask_role():
    print("\n可选角色：")
    for key, label in ROLE_OPTIONS.items():
        print(f"- {key}: {label}")

    role = input("请输入角色，例如 legal / finance / market / product：").strip().lower()

    if role == "":
        return "default"

    if role not in ROLE_OPTIONS:
        print("未识别角色，使用 default。")
        return "default"

    return role


def load_or_empty(path, columns):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame(columns=columns)


def append_row(path, columns, row):
    df = load_or_empty(path, columns)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return df


def input_manual_prior():
    print("\n=== 业务先验 manual_prior ===")
    print("这部分影响内容适配、执行落地、授权成本、法务风险。")
    print("如果不填，notebook 会使用默认中性值。")

    while True:
        target = ask_text("\n候选 IP 名称，例如 Porsche：")
        if target == "":
            print("候选 IP 不能为空。")
            continue

        row = {
            "target_interest": target,
            "content_fit_prior": ask_score("内容适配度 0-100，默认 60：", default=60),
            "execution_prior": ask_score("执行可行性 0-100，默认 60：", default=60),
            "budget_risk": ask_risk("授权成本风险"),
            "legal_risk": ask_risk("法务/品牌审核风险"),
            "competitor_used": ask_yes_no("竞品近期是否用过这个 IP？", default=False)
        }

        df = append_row(MANUAL_PRIOR_PATH, MANUAL_PRIOR_COLUMNS, row)

        print(f"\n已写入 {MANUAL_PRIOR_PATH}")
        print(df.tail())

        if not ask_yes_no("继续录入另一个 IP 的业务先验？", default=False):
            break


def input_human_score():
    print("\\n=== 人工评分 human_scores ===")
    print("这部分是市场/产品/法务/财务/商务/社区等角色的主观评分。")
    print("如果不填，notebook 最终分默认等于自动分。")

    while True:
        target = ask_text("\\n候选 IP 名称，例如 Porsche：")
        if target == "":
            print("候选 IP 不能为空。")
            continue

        evaluator = ask_text("评分人名称/代号，例如 legal_A：", default="anonymous")
        role = ask_role()

        row = {
            "target_interest": target,
            "evaluator": evaluator,
            "role": role
        }

        print("\n请给以下维度打分，范围 0-100：")
        for field in HUMAN_SCORE_FIELDS:
            row[field] = ask_score(f"{field}：")

        row["confidence"] = ask_confidence()

        df = append_row(HUMAN_SCORES_PATH, HUMAN_SCORE_COLUMNS, row)

        print(f"\n已写入 {HUMAN_SCORES_PATH}")
        print(df.tail())

        if not ask_yes_no("继续录入另一条人工评分？", default=False):
            break


def main():
    print("====================================")
    print("IP 联动评分录入终端")
    print("====================================")
    print("会生成两个文件：")
    print(f"- {MANUAL_PRIOR_PATH}")
    print(f"- {HUMAN_SCORES_PATH}")
    print("notebook 后面的第 14、15 格会自动读取它们。")

    if ask_yes_no("\n是否录入业务先验 manual_prior？", default=True):
        input_manual_prior()
    else:
        print("跳过 manual_prior。notebook 会使用默认中性值。")

    if ask_yes_no("\n是否录入人工评分 human_scores？", default=True):
        input_human_score()
    else:
        print("跳过 human_scores。最终分默认等于自动分。")

    print("\n完成。")


if __name__ == "__main__":
    main()
'''

with open("ip_score_input_cli.py", "w", encoding="utf-8") as f:
    f.write(cli_code)

print("已生成 ip_score_input_cli.py")
print("下面开始录入：")

!python ip_score_input_cli.py

已生成 ip_score_input_cli.py
下面开始录入：
IP 联动评分录入终端
会生成两个文件：
- manual_prior.csv
- human_scores.csv
notebook 后面的第 14、15 格会自动读取它们。

是否录入业务先验 manual_prior？ [Y/n]: no
跳过 manual_prior。notebook 会使用默认中性值。

是否录入人工评分 human_scores？ [Y/n]: no
跳过 human_scores。最终分默认等于自动分。

完成。现在从第 14 格开始继续往下跑。


## 14. 人工业务先验

这些是业务同事要填的。

它们不是爬虫自动得出的：

- 内容适配
- 执行难度
- 授权成本风险
- 法务风险
- 竞品是否近期用过


In [59]:
# =========================
# 14. 人工业务先验 manual_prior
# =========================
# 这部分可以由终端脚本生成 manual_prior.csv
# 如果没有 manual_prior.csv，就给每个候选 IP 一个默认中性值

from pathlib import Path

MANUAL_PRIOR_COLUMNS = [
    "target_interest",
    "content_fit_prior",
    "execution_prior",
    "budget_risk",
    "legal_risk",
    "competitor_used"
]

manual_prior_path = Path("manual_prior.csv")

if manual_prior_path.exists():
    manual_prior = pd.read_csv(manual_prior_path)
else:
    manual_prior = pd.DataFrame([
        {
            "target_interest": ip["interest"],
            "content_fit_prior": 60,
            "execution_prior": 60,
            "budget_risk": "mid",
            "legal_risk": "mid",
            "competitor_used": False
        }
        for ip in target_ips
    ])

# 类型清洗
manual_prior["content_fit_prior"] = pd.to_numeric(
    manual_prior["content_fit_prior"], errors="coerce"
).fillna(60).clip(0, 100)

manual_prior["execution_prior"] = pd.to_numeric(
    manual_prior["execution_prior"], errors="coerce"
).fillna(60).clip(0, 100)

manual_prior["budget_risk"] = manual_prior["budget_risk"].fillna("mid")
manual_prior["legal_risk"] = manual_prior["legal_risk"].fillna("mid")

manual_prior["competitor_used"] = manual_prior["competitor_used"].astype(str).str.lower().isin(
    ["true", "1", "yes", "y", "是"]
)

display(manual_prior)

,target_interest,content_fit_prior,execution_prior,budget_risk,legal_risk,competitor_used
0,Porsche,60,60,mid,mid,False


## 15. 人工评分表

这部分可以先不填真实人名，demo 里先保留结构。

如果没有人工评分，后面会让 human score 默认等于自动 baseline score。


In [60]:
# =========================
# 15. 人工评分表 human_scores
# =========================
# 这部分可以由终端脚本生成 human_scores.csv
# 如果没有 human_scores.csv，就保持空表
# 空表时，最终分默认等于自动分

from pathlib import Path

HUMAN_SCORE_COLUMNS = [
    "target_interest",
    "evaluator",
    "role",
    "人工用户匹配",
    "人工内容适配",
    "人工商业潜力",
    "人工传播潜力",
    "人工风险安全",
    "人工执行可行",
    "confidence"
]

human_score_path = Path("human_scores.csv")

if human_score_path.exists():
    human_scores = pd.read_csv(human_score_path)
else:
    human_scores = pd.DataFrame(columns=HUMAN_SCORE_COLUMNS)


role_weights = {
    "market": 1.20,
    "product": 1.25,
    "bizdev": 1.00,
    "legal": 1.15,
    "finance": 1.10,
    "community": 1.10,
    "default": 1.00
}

human_dimension_weights = {
    "人工用户匹配": 0.25,
    "人工内容适配": 0.20,
    "人工商业潜力": 0.20,
    "人工风险安全": 0.15,
    "人工传播潜力": 0.10,
    "人工执行可行": 0.10,
}


def compute_human_scores(human_scores):
    if human_scores is None or human_scores.empty:
        return pd.DataFrame(columns=[
            "target_interest",
            "human_score_weighted",
            "human_evaluator_count",
            "human_score_std"
        ])

    df = human_scores.copy()

    for c in human_dimension_weights:
        df[c] = pd.to_numeric(df[c], errors="coerce").clip(0, 100)

    df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce").fillna(1).clip(0, 1)

    df["human_score_individual"] = 0
    for col, w in human_dimension_weights.items():
        df["human_score_individual"] += df[col] * w

    df["role_weight"] = df["role"].map(role_weights).fillna(role_weights["default"])
    df["final_person_weight"] = df["role_weight"] * df["confidence"]

    rows = []

    for ip, g in df.groupby("target_interest"):
        total_w = g["final_person_weight"].sum()

        weighted = (
            (g["human_score_individual"] * g["final_person_weight"]).sum() / total_w
            if total_w > 0 else g["human_score_individual"].mean()
        )

        rows.append({
            "target_interest": ip,
            "human_score_weighted": round(weighted, 2),
            "human_evaluator_count": len(g),
            "human_score_std": round(g["human_score_individual"].std(ddof=0), 2)
        })

    return pd.DataFrame(rows)


human_agg_df = compute_human_scores(human_scores)

display(human_scores)
display(human_agg_df)

,target_interest,evaluator,role,人工用户匹配,人工内容适配,人工商业潜力,人工传播潜力,人工风险安全,人工执行可行,confidence


,target_interest,human_score_weighted,human_evaluator_count,human_score_std


## 16. 评分：Pre-campaign Baseline Fit

这是预测未来联动时最重要的分。

只使用 baseline 时间窗的数据，不使用活动期热度。

满分 100：

- Audience Overlap /25
- Content Fit /20
- Commercial Potential /20
- Reputation Safety /15
- Community Spread /10
- Execution Feasibility /10


In [61]:
def get_feature(feature_df, target, window):
    sub = feature_df[(feature_df["target_interest"] == target) & (feature_df["window"] == window)]
    if sub.empty:
        return pd.Series(dtype="object")
    return sub.iloc[0]

def get_creator_feature(creator_feature_df, target, window):
    sub = creator_feature_df[(creator_feature_df["target_interest"] == target) & (creator_feature_df["window"] == window)]
    if sub.empty:
        return pd.Series(dtype="object")
    return sub.iloc[0]

baseline_score_rows = []

for ip in target_ips:
    target = ip["interest"]

    overlap = baseline_overlap_df[baseline_overlap_df["target_interest"] == target]
    overlap = overlap.iloc[0] if not overlap.empty else pd.Series(dtype="object")

    feat = get_feature(window_feature_df, target, "baseline")
    creator = get_creator_feature(creator_feature_df, target, "baseline")
    prior = manual_prior[manual_prior["target_interest"] == target].iloc[0]

    # 1. Audience Overlap /25
    audience_overlap = (
        safe_num(overlap.get("core_overlap_rate", 0)) * 8
        + safe_num(overlap.get("weighted_core_overlap_rate", 0)) * 7
        + safe_num(overlap.get("active_overlap_rate", 0)) * 3
        + safe_num(overlap.get("high_engagement_overlap_rate", 0)) * 3
        + min(safe_num(overlap.get("lift_core_vs_base", 0)), 5) / 5 * 4
    )
    audience_overlap = min(25, max(0, audience_overlap))

    # 2. Content Fit /20
    content_index = (
        safe_num(prior.get("content_fit_prior", 60)) * 0.60
        + safe_num(feat.get("positive_rate", 0)) * 18
        + (1 - safe_num(feat.get("negative_rate", 0))) * 11
        + (1 - safe_num(feat.get("fit_risk_rate", 0))) * 11
    )
    content_fit = min(20, max(0, content_index / 100 * 20))

    # 3. Commercial Potential /20
    commercial_index = (
        safe_num(feat.get("purchase_intent_rate", 0)) * 40
        + log_score_0_100([safe_num(feat.get("total_engagement", 0))]).iloc[0] * 0.20
        + log_score_0_100([safe_num(creator.get("creator_total_followers", 0))]).iloc[0] * 0.20
        + log_score_0_100([safe_num(creator.get("creator_total_views", 0))]).iloc[0] * 0.20
    )
    commercial = min(20, max(0, commercial_index / 100 * 20))

    # 4. Reputation Safety /15
    safety_index = (
        (1 - safe_num(feat.get("negative_rate", 0))) * 45
        + (1 - safe_num(feat.get("fit_risk_rate", 0))) * 40
        + 15
        - risk_penalty(prior.get("legal_risk", "mid")) * 0.35
    )
    reputation_safety = min(15, max(0, safety_index / 100 * 15))

    # 5. Community Spread /10
    spread_index = (
        safe_num(feat.get("ugc_rate", 0)) * 35
        + safe_num(feat.get("creator_discussion_rate", 0)) * 20
        + log_score_0_100([safe_num(feat.get("total_shares", 0))]).iloc[0] * 0.20
        + log_score_0_100([safe_num(creator.get("creator_total_views", 0))]).iloc[0] * 0.25
    )
    community_spread = min(10, max(0, spread_index / 100 * 10))

    # 6. Execution Feasibility /10
    execution_index = (
        safe_num(prior.get("execution_prior", 60))
        - risk_penalty(prior.get("budget_risk", "mid")) * 0.55
        - (8 if bool(prior.get("competitor_used", False)) else 0)
    )
    execution = min(10, max(0, execution_index / 100 * 10))

    final_auto = audience_overlap + content_fit + commercial + reputation_safety + community_spread + execution

    baseline_score_rows.append({
        "target_interest": target,
        "pre_campaign_baseline_score_auto": round(final_auto, 2),
        "pre_campaign_baseline_grade_auto": grade(final_auto),

        "Audience Overlap /25": round(audience_overlap, 2),
        "Content Fit /20": round(content_fit, 2),
        "Commercial Potential /20": round(commercial, 2),
        "Reputation Safety /15": round(reputation_safety, 2),
        "Community Spread /10": round(community_spread, 2),
        "Execution Feasibility /10": round(execution, 2),

        "core_overlap_rate": safe_num(overlap.get("core_overlap_rate", 0)),
        "weighted_core_overlap_rate": safe_num(overlap.get("weighted_core_overlap_rate", 0)),
        "active_overlap_rate": safe_num(overlap.get("active_overlap_rate", 0)),
        "high_engagement_overlap_rate": safe_num(overlap.get("high_engagement_overlap_rate", 0)),
        "lift_core_vs_base": safe_num(overlap.get("lift_core_vs_base", 0)),

        "baseline_purchase_intent_rate": safe_num(feat.get("purchase_intent_rate", 0)),
        "baseline_positive_rate": safe_num(feat.get("positive_rate", 0)),
        "baseline_negative_rate": safe_num(feat.get("negative_rate", 0)),
        "baseline_fit_risk_rate": safe_num(feat.get("fit_risk_rate", 0)),
        "baseline_ugc_rate": safe_num(feat.get("ugc_rate", 0)),
        "baseline_creator_discussion_rate": safe_num(feat.get("creator_discussion_rate", 0)),

        "baseline_relevant_content_count": safe_num(feat.get("relevant_content_count", 0)),
        "baseline_rows": len(baseline_df)
    })

baseline_score_df = pd.DataFrame(baseline_score_rows)

display(baseline_score_df)


,target_interest,pre_campaign_baseline_score_auto,pre_campaign_baseline_grade_auto,Audience Overlap /25,Content Fit /20,Commercial Potential /20,Reputation Safety /15,Community Spread /10,Execution Feasibility /10,core_overlap_rate,weighted_core_overlap_rate,active_overlap_rate,high_engagement_overlap_rate,lift_core_vs_base,baseline_purchase_intent_rate,baseline_positive_rate,baseline_negative_rate,baseline_fit_risk_rate,baseline_ugc_rate,baseline_creator_discussion_rate,baseline_relevant_content_count,baseline_rows
0,Porsche,62.84,C,15.6,11.6,14.0,14.21,2.25,5.17,0.5,0.628563,0.333333,1.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,9


## 17. 人工加权预测分

这一步只融合 baseline 预测分，不融合活动期表现。

默认：

- 自动分 70%
- 人工分 30%


In [62]:
# =========================
# 17. 人工加权预测分
# =========================
# 没有人打分时：最终分 = 自动分
# 有人打分时：最终分 = 自动分 × 70% + 人工分 × 30%

AUTO_WEIGHT = 0.70

baseline_score_df = baseline_score_df.merge(human_agg_df, on="target_interest", how="left")

baseline_score_df["has_human_score"] = baseline_score_df["human_score_weighted"].notna()

baseline_score_df["human_score_weighted"] = baseline_score_df["human_score_weighted"].fillna(
    baseline_score_df["pre_campaign_baseline_score_auto"]
)

baseline_score_df["human_evaluator_count"] = baseline_score_df["human_evaluator_count"].fillna(0).astype(int)

if "human_score_std" in baseline_score_df.columns:
    baseline_score_df["human_score_std"] = baseline_score_df["human_score_std"].fillna(0)
else:
    baseline_score_df["human_score_std"] = 0

baseline_score_df["pre_campaign_baseline_score_human_weighted"] = np.where(
    baseline_score_df["has_human_score"],
    baseline_score_df["pre_campaign_baseline_score_auto"] * AUTO_WEIGHT
    + baseline_score_df["human_score_weighted"] * (1 - AUTO_WEIGHT),
    baseline_score_df["pre_campaign_baseline_score_auto"]
).round(2)

baseline_score_df["pre_campaign_baseline_grade_human_weighted"] = baseline_score_df[
    "pre_campaign_baseline_score_human_weighted"
].apply(grade)

baseline_score_df["baseline_recommended_action"] = baseline_score_df[
    "pre_campaign_baseline_grade_human_weighted"
].apply(action_for_grade)

display(baseline_score_df[[
    "target_interest",
    "pre_campaign_baseline_score_auto",
    "has_human_score",
    "human_score_weighted",
    "human_evaluator_count",
    "pre_campaign_baseline_score_human_weighted",
    "pre_campaign_baseline_grade_human_weighted",
    "baseline_recommended_action"
]])

/tmp/ipykernel_2941/4065767244.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  baseline_score_df["human_score_weighted"] = baseline_score_df["human_score_weighted"].fillna(
/tmp/ipykernel_2941/4065767244.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  baseline_score_df["human_evaluator_count"] = baseline_score_df["human_evaluator_count"].fillna(0).astype(int)
/tmp/ipykernel_2941/4065767244.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects

,target_interest,pre_campaign_baseline_score_auto,has_human_score,human_score_weighted,human_evaluator_count,pre_campaign_baseline_score_human_weighted,pre_campaign_baseline_grade_human_weighted,baseline_recommended_action
0,Porsche,62.84,False,62.84,0,62.84,C,低优先级，仅低成本时考虑


## 18. Campaign Performance Score

这部分只看活动期表现。

它不代表“预测能力”，只代表活动实际打起来没有。


In [63]:
campaign_rows = []

for ip in target_ips:
    target = ip["interest"]
    feat = get_feature(window_feature_df, target, "campaign")
    creator = get_creator_feature(creator_feature_df, target, "campaign")

    heat_index = (
        safe_num(feat.get("event_hit_rate", 0)) * 30
        + safe_num(feat.get("purchase_intent_rate", 0)) * 20
        + safe_num(feat.get("positive_rate", 0)) * 15
        + safe_num(feat.get("ugc_rate", 0)) * 10
        + safe_num(feat.get("creator_discussion_rate", 0)) * 10
        + log_score_0_100([safe_num(feat.get("total_engagement", 0))]).iloc[0] * 0.10
        + log_score_0_100([safe_num(creator.get("creator_total_views", 0))]).iloc[0] * 0.05
    )

    risk = (
        safe_num(feat.get("negative_rate", 0)) * 15
        + safe_num(feat.get("fit_risk_rate", 0)) * 15
    )

    score = min(100, max(0, heat_index - risk))

    campaign_rows.append({
        "target_interest": target,
        "campaign_performance_score": round(score, 2),
        "campaign_performance_grade": grade(score),
        "campaign_relevant_content_count": safe_num(feat.get("relevant_content_count", 0)),
        "campaign_event_hit_rate": safe_num(feat.get("event_hit_rate", 0)),
        "campaign_purchase_intent_rate": safe_num(feat.get("purchase_intent_rate", 0)),
        "campaign_positive_rate": safe_num(feat.get("positive_rate", 0)),
        "campaign_negative_rate": safe_num(feat.get("negative_rate", 0)),
        "campaign_fit_risk_rate": safe_num(feat.get("fit_risk_rate", 0)),
        "campaign_ugc_rate": safe_num(feat.get("ugc_rate", 0)),
        "campaign_creator_discussion_rate": safe_num(feat.get("creator_discussion_rate", 0)),
        "campaign_total_engagement": safe_num(feat.get("total_engagement", 0)),
        "campaign_total_views": safe_num(feat.get("total_views", 0)),
        "campaign_rows": len(campaign_df)
    })

campaign_score_df = pd.DataFrame(campaign_rows)
display(campaign_score_df)


,target_interest,campaign_performance_score,campaign_performance_grade,campaign_relevant_content_count,campaign_event_hit_rate,campaign_purchase_intent_rate,campaign_positive_rate,campaign_negative_rate,campaign_fit_risk_rate,campaign_ugc_rate,campaign_creator_discussion_rate,campaign_total_engagement,campaign_total_views,campaign_rows
0,Porsche,37.5,D,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,11545.0,89800.0,11


## 19. Post-campaign Retention / Decay Score

这部分看活动结束后还有没有残留讨论。

不是越热越好，而是看活动后是否还能留下内容、二创、正向反馈。


In [64]:
post_rows = []

for ip in target_ips:
    target = ip["interest"]
    camp = get_feature(window_feature_df, target, "campaign")
    post = get_feature(window_feature_df, target, "post")

    campaign_per_day = safe_div(safe_num(camp.get("relevant_content_count", 0)), campaign_days)
    post_per_day = safe_div(safe_num(post.get("relevant_content_count", 0)), post_days)

    retention_ratio = safe_div(post_per_day, campaign_per_day)
    retention_ratio_capped = min(retention_ratio, 1.5)

    retention_score = (
        retention_ratio_capped / 1.5 * 45
        + safe_num(post.get("positive_rate", 0)) * 20
        + safe_num(post.get("ugc_rate", 0)) * 15
        + safe_num(post.get("creator_discussion_rate", 0)) * 10
        + (1 - safe_num(post.get("negative_rate", 0))) * 10
    )
    retention_score = min(100, max(0, retention_score))

    post_rows.append({
        "target_interest": target,
        "post_campaign_retention_score": round(retention_score, 2),
        "post_campaign_retention_grade": grade(retention_score),
        "campaign_relevant_per_day": round(campaign_per_day, 4),
        "post_relevant_per_day": round(post_per_day, 4),
        "retention_ratio": round(retention_ratio, 4),
        "post_relevant_content_count": safe_num(post.get("relevant_content_count", 0)),
        "post_positive_rate": safe_num(post.get("positive_rate", 0)),
        "post_negative_rate": safe_num(post.get("negative_rate", 0)),
        "post_fit_risk_rate": safe_num(post.get("fit_risk_rate", 0)),
        "post_ugc_rate": safe_num(post.get("ugc_rate", 0)),
        "post_creator_discussion_rate": safe_num(post.get("creator_discussion_rate", 0)),
        "post_rows": len(post_df)
    })

post_score_df = pd.DataFrame(post_rows)
display(post_score_df)


,target_interest,post_campaign_retention_score,post_campaign_retention_grade,campaign_relevant_per_day,post_relevant_per_day,retention_ratio,post_relevant_content_count,post_positive_rate,post_negative_rate,post_fit_risk_rate,post_ugc_rate,post_creator_discussion_rate,post_rows
0,Porsche,10.0,D,0.0714,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## 20. 总结果表

这张表把三类分数放在一起：

- 联动前预测分
- 活动期表现分
- 活动后留存分

注意：这三类不要混成一个分。它们回答的问题不同。


In [52]:
final_df = (
    baseline_score_df
    .merge(campaign_score_df, on="target_interest", how="left")
    .merge(post_score_df, on="target_interest", how="left")
)

display(final_df)


,target_interest,pre_campaign_baseline_score_auto,pre_campaign_baseline_grade_auto,Audience Overlap /25,Content Fit /20,Commercial Potential /20,Reputation Safety /15,Community Spread /10,Execution Feasibility /10,core_overlap_rate,weighted_core_overlap_rate,active_overlap_rate,high_engagement_overlap_rate,lift_core_vs_base,baseline_purchase_intent_rate,baseline_positive_rate,baseline_negative_rate,baseline_fit_risk_rate,baseline_ugc_rate,baseline_creator_discussion_rate,baseline_relevant_content_count,baseline_rows,human_score_weighted,human_evaluator_count,human_score_std,has_human_score,pre_campaign_baseline_score_human_weighted,pre_campaign_baseline_grade_human_weighted,baseline_recommended_action,campaign_performance_score,campaign_performance_grade,campaign_relevant_content_count,campaign_event_hit_rate,campaign_purchase_intent_rate,campaign_positive_rate,campaign_negative_rate,campaign_fit_risk_rate,campaign_ugc_rate,campaign_creator_discussion_rate,campaign_total_engagement,campaign_total_views,campaign_rows,post_campaign_retention_score,post_campaign_retention_grade,campaign_relevant_per_day,post_relevant_per_day,retention_ratio,post_relevant_content_count,post_positive_rate,post_negative_rate,post_fit_risk_rate,post_ugc_rate,post_creator_discussion_rate,post_rows
0,Porsche,62.84,C,15.6,11.6,14.0,14.21,2.25,5.17,0.5,0.628563,0.333333,1.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,9,62.84,0,0,False,62.84,C,低优先级，仅低成本时考虑,37.5,D,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,11545.0,89800.0,11,10.0,D,0.0714,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## 21. 中文汇总表

这张表适合直接给别人看。


In [65]:
cn_rows = []

for _, r in final_df.iterrows():
    cn_rows.append({
        "候选IP": r["target_interest"],

        "联动前预测分-自动": r["pre_campaign_baseline_score_auto"],
        "联动前预测分-人工加权": r["pre_campaign_baseline_score_human_weighted"],
        "联动前预测等级": r["pre_campaign_baseline_grade_human_weighted"],
        "预测推荐动作": r["baseline_recommended_action"],

        "活动期表现分": r["campaign_performance_score"],
        "活动期表现等级": r["campaign_performance_grade"],

        "活动后留存分": r["post_campaign_retention_score"],
        "活动后留存等级": r["post_campaign_retention_grade"],

        "用户重合 /25": r["Audience Overlap /25"],
        "内容适配 /20": r["Content Fit /20"],
        "商业潜力 /20": r["Commercial Potential /20"],
        "舆情安全 /15": r["Reputation Safety /15"],
        "社区传播 /10": r["Community Spread /10"],
        "执行落地 /10": r["Execution Feasibility /10"],

        "核心用户重合率": pct(r["core_overlap_rate"]),
        "加权核心重合率": pct(r["weighted_core_overlap_rate"]),
        "活跃用户重合率": pct(r["active_overlap_rate"]),
        "高互动用户重合率": pct(r["high_engagement_overlap_rate"]),
        "Lift": f"{r['lift_core_vs_base']:.2f}x",

        "baseline付费意图率": pct(r["baseline_purchase_intent_rate"]),
        "baseline负面率": pct(r["baseline_negative_rate"]),
        "baseline违和风险率": pct(r["baseline_fit_risk_rate"]),
        "baseline二创率": pct(r["baseline_ugc_rate"]),

        "活动期事件命中率": pct(r["campaign_event_hit_rate"]),
        "活动期付费意图率": pct(r["campaign_purchase_intent_rate"]),
        "活动期正向率": pct(r["campaign_positive_rate"]),
        "活动期负面率": pct(r["campaign_negative_rate"]),
        "活动期违和风险率": pct(r["campaign_fit_risk_rate"]),
        "活动期二创率": pct(r["campaign_ugc_rate"]),

        "活动期日均相关内容": r["campaign_relevant_per_day"],
        "活动后日均相关内容": r["post_relevant_per_day"],
        "活动后/活动期留存比": r["retention_ratio"],

        "baseline样本量": int(r["baseline_rows"]),
        "campaign样本量": int(r["campaign_rows"]),
        "post样本量": int(r["post_rows"]),

        "解释口径": "预测分只看联动前；活动表现只看活动期；活动后留存看衰减。collab/联动词不进入用户重合。"
    })

cn_output_df = pd.DataFrame(cn_rows)
display(cn_output_df)


,候选IP,联动前预测分-自动,联动前预测分-人工加权,联动前预测等级,预测推荐动作,活动期表现分,活动期表现等级,活动后留存分,活动后留存等级,用户重合 /25,内容适配 /20,商业潜力 /20,舆情安全 /15,社区传播 /10,执行落地 /10,核心用户重合率,加权核心重合率,活跃用户重合率,高互动用户重合率,Lift,baseline付费意图率,baseline负面率,baseline违和风险率,baseline二创率,活动期事件命中率,活动期付费意图率,活动期正向率,活动期负面率,活动期违和风险率,活动期二创率,活动期日均相关内容,活动后日均相关内容,活动后/活动期留存比,baseline样本量,campaign样本量,post样本量,解释口径
0,Porsche,62.84,62.84,C,低优先级，仅低成本时考虑,37.5,D,10.0,D,15.6,11.6,14.0,14.21,2.25,5.17,50.0%,62.9%,33.3%,100.0%,4.00x,100.0%,0.0%,0.0%,0.0%,0.0%,100.0%,0.0%,0.0%,0.0%,0.0%,0.0714,0.0,0.0,9,11,0,预测分只看联动前；活动表现只看活动期；活动后留存看衰减。collab/联动词不进入用户重合。


## 22. 导出 CSV / Excel / HTML

会导出三个文件：

- 中文汇总 CSV
- 多 sheet Excel
- 美化版 HTML


In [54]:
csv_out = "IP联动评分_v7_promax_中文汇总.csv"
xlsx_out = "IP联动评分_v7_promax_完整结果.xlsx"
html_out = "IP联动评分_v7_promax_报告.html"

cn_output_df.to_csv(csv_out, index=False, encoding="utf-8-sig")

def clean_excel_df(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            try:
                if getattr(df[c].dt, "tz", None) is not None:
                    df[c] = df[c].dt.tz_localize(None)
            except Exception:
                df[c] = df[c].astype(str)
        else:
            df[c] = df[c].apply(
                lambda x: x.tz_localize(None) if isinstance(x, pd.Timestamp) and x.tzinfo is not None else x
            )
    return df

with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
    clean_excel_df(cn_output_df).to_excel(writer, index=False, sheet_name="中文汇总")
    clean_excel_df(final_df).to_excel(writer, index=False, sheet_name="final_all_scores")
    clean_excel_df(baseline_score_df).to_excel(writer, index=False, sheet_name="baseline_prediction")
    clean_excel_df(campaign_score_df).to_excel(writer, index=False, sheet_name="campaign_performance")
    clean_excel_df(post_score_df).to_excel(writer, index=False, sheet_name="post_retention")
    clean_excel_df(window_feature_df).to_excel(writer, index=False, sheet_name="window_features")
    clean_excel_df(baseline_overlap_df).to_excel(writer, index=False, sheet_name="baseline_overlap")
    clean_excel_df(baseline_user_strength_df).to_excel(writer, index=False, sheet_name="user_strength")
    clean_excel_df(baseline_signal_df).to_excel(writer, index=False, sheet_name="baseline_signals")
    clean_excel_df(creator_feature_df).to_excel(writer, index=False, sheet_name="creator_features")
    clean_excel_df(manual_prior).to_excel(writer, index=False, sheet_name="manual_prior")
    clean_excel_df(human_scores).to_excel(writer, index=False, sheet_name="human_scores")
    clean_excel_df(content_df).to_excel(writer, index=False, sheet_name="content_raw")

html_table = cn_output_df.to_html(index=False, escape=False)

html = f'''
<!DOCTYPE html>
<html lang="zh-CN">
<head>
<meta charset="UTF-8">
<title>IP联动评分 v7 Pro Max 报告</title>
<style>
body {{
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Microsoft YaHei", Arial, sans-serif;
    background: #f6f7fb;
    margin: 32px;
    color: #111827;
}}
h1 {{ margin-bottom: 8px; }}
.subtitle {{
    color: #6b7280;
    margin-bottom: 24px;
    line-height: 1.7;
}}
.card {{
    background: white;
    border-radius: 16px;
    padding: 24px;
    box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
    overflow-x: auto;
}}
table {{
    border-collapse: collapse;
    min-width: 2600px;
    font-size: 14px;
}}
th {{
    background: #111827;
    color: white;
    padding: 10px;
    text-align: left;
}}
td {{
    border-bottom: 1px solid #e5e7eb;
    padding: 10px;
    vertical-align: top;
}}
tr:hover {{ background: #f9fafb; }}
.note {{
    margin-top: 18px;
    color: #6b7280;
    font-size: 13px;
    line-height: 1.7;
}}
</style>
</head>
<body>
<h1>IP联动评分 v7 Pro Max 报告</h1>
<div class="subtitle">
这版把联动前预测、活动期表现、活动后留存分开计算；同时保留用户重合、内容适配、商业潜力、舆情安全、社区传播、执行落地等细项。
</div>
<div class="card">
{html_table}
<div class="note">
说明：预测分只看 baseline 时间窗；活动表现只看 campaign 时间窗；活动后留存只看 post 时间窗。
collab / 联动 / NTEPorsche 等事件词不进入用户重合度，只进入活动表现。
</div>
</div>
</body>
</html>
'''

with open(html_out, "w", encoding="utf-8") as f:
    f.write(html)

try:
    from google.colab import files
    files.download(csv_out)
    files.download(xlsx_out)
    files.download(html_out)
except Exception:
    print("已保存:", csv_out)
    print("已保存:", xlsx_out)
    print("已保存:", html_out)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 这版和 v7 简化版相比，多了什么

- 恢复 6 大 baseline 预测维度，总分 100
  - Audience Overlap /25
  - Content Fit /20
  - Commercial Potential /20
  - Reputation Safety /15
  - Community Spread /10
  - Execution Feasibility /10
- 保留人工评分和人工加权分
- 保留 creator / KOL 弱特征入口
- 保留活动期表现分
- 保留活动后留存分
- 保留 Excel 多 sheet 导出
- 保留 HTML 中文报告
- 保留时间窗逻辑，避免回测污染预测分
